Привет, Дмитрий! 

Меня зовут Светлана Медведева и я буду проверять Твою работу. Предлагаю общаться на "ты". Если Тебе такой вариант не удобен. то пожалуйста дай знать. Я сразу перейду на "Вы".

При обнаружении у Тебя в работе ошибки, в первый раз я лишь укажу на ее наличие и у Тебя будет возможность самому найти её и исправить. В реальной работе твой начальник будет поступать также, а я пытаюсь подготовить тебя именно к работе Data Scientist. Однако, если ты пока не справишься с такой задачей - при следующей проверке я дам более точную подсказку.

Просьба при доработке работы оставлять мои комментарии без изменений.

Комментарии я разделяю на следующие категории:

<div class="alert alert-block alert-success">
В случае если всё верно!
</div>


<div class="alert alert-block alert-warning">
В случае если можно что-то доработать, но эта доработка не критична или если есть варианты улучшения работы.
</div>

<div class="alert alert-block alert-danger">
Критичные замечания. Если бы проект сдавался за несколько итераций, то с красными замечаниями проект не был бы принят.
</div>

<div class="alert alert-block alert-info">
Дополнительные материалы, выходящие за рамки программы.
</div>

## Ревью:

У Тебя в целом неплохая работа, но можно улучшить качество модели за счёт подбора гиперпараметров и повышения качества данных. Кроме того, можно рассмотреть другие подходы к решению задачи многоклассовой классификации.

Успехов!

In [1]:
import os

In [ ]:
!pip install sentence_transformers

In [193]:
import pathlib
import random
import pandas as pd
import numpy as np
import sys
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict
from sentence_transformers import SentenceTransformer, util
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from transformers import AutoTokenizer, AutoModelForSequenceClassification,pipeline

from tqdm import tqdm

from catboost import CatBoostClassifier
from sklearn.metrics import (
    f1_score, 
    accuracy_score,
    classification_report, 
)
ROOT_DIR = pathlib.Path().absolute().parent
DATA_DIR = ROOT_DIR / "Samokat.tech_Workshop/Data"

RANDOM_SEED = 42

<div class="alert alert-block alert-success">

Отлично, что сохранил в отдельную переменную значение для random_state. Важно зафиксировать random_state для воспроизводимости результатов.
</div>

In [59]:
tqdm.pandas()

## Загрузка и обзор данных

In [ ]:
df_trends = pd.read_csv(DATA_DIR / "trends_description.csv")
df = pd.read_csv(DATA_DIR / "train.csv")

In [ ]:
df_trends['explanation'][2]

In [ ]:
df.head()

## Обучение моделей

### Baseline-модель

### Предобработка данных

In [105]:
df.head()

,index,assessment,tags,text,trend_id_res0,trend_id_res1,trend_id_res2,trend_id_res3,trend_id_res4,trend_id_res5,...,trend_id_res40,trend_id_res41,trend_id_res42,trend_id_res43,trend_id_res44,trend_id_res45,trend_id_res46,trend_id_res47,trend_id_res48,trend_id_res49
0,5652,6.0,"{ASSORTMENT,PROMOTIONS,DELIVERY}","Маленький выбор товаров, хотелось бы ассортиме...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,18092,4.0,"{ASSORTMENT,PRICE,PRODUCTS_QUALITY,DELIVERY}",Быстро,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13845,6.0,"{DELIVERY,PROMOTIONS,PRICE,ASSORTMENT,SUPPORT}",Доставка постоянно задерживается,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,25060,6.0,"{PRICE,PROMOTIONS,ASSORTMENT}",Наценка и ассортимент расстраивают,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,15237,5.0,"{ASSORTMENT,PRODUCTS_QUALITY,PROMOTIONS,CATALO...",Доставка просто 👍,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<div class="alert alert-block alert-danger">

После загрузки данных надо их проанализировать на наличие пропусков, дубликатов и проверить можно ли оптимизировтаь тип данных.

При просмотре общей информации о данных имеет смысл проверить корректность типов данных. Можно ли оптимизировать тип данных? Для больших датасетов оптимизация типов данных позволяет сократить объём занимаемой оперативной памяти.

Подробнее: https://towardsdatascience.com/seven-killer-memory-optimization-techniques-every-pandas-user-should-know-64707348ab20
</div>

<div class="alert alert-block alert-warning">

Для работы с текстом нужно выполнить очистку и лемматизацию, а после этого векторизацию.

Дополнительно по лемматизации можно посмотреть: https://webdevblog.ru/podhody-lemmatizacii-s-primerami-v-python/
    
P. S. Вижу, что для следующих моделей эти шаги выполняешь)
</div>

In [106]:
X, y = df[["text"]], df[[f"trend_id_res{i}" for i in range(50)]]
X = X.astype("str").copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_SEED)
print(f"X_train.shape is {X_train.shape}")
print(f"y_train.shape is {y_train.shape}")
print(f"X_test.shape is {X_test.shape}")
print(f"y_test.shape is {y_test.shape}")

X_train.shape is (6966, 1)
y_train.shape is (6966, 50)
X_test.shape is (1742, 1)
y_test.shape is (1742, 50)


<div class="alert alert-block alert-success">

Молодец, что проверил размер выборок 👍👍👍
</div>

###  Проверка качества на тречнировчном датасете

In [107]:
preprocessor = ColumnTransformer(
    [
        ("vetorizer", TfidfVectorizer(analyzer="char_wb", ngram_range = (1,3)), "text")
    ],                         
    remainder = "passthrough"
)

pipeline_multiout = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", MultiOutputClassifier(LogisticRegression(max_iter = 10_000))),
    ]
)
display(pipeline_multiout)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('vetorizer',
                                                  TfidfVectorizer(analyzer='char_wb',
                                                                  ngram_range=(1,
                                                                               3)),
                                                  'text')])),
                ('clf',
                 MultiOutputClassifier(estimator=LogisticRegression(max_iter=10000)))])

<div class="alert alert-block alert-success">

Да, использовать Pipeline для обучения модели это отличная идея.

Дополнительно можно посмотреть: https://www.techtarget.com/searchenterpriseai/tip/Learn-how-to-create-a-machine-learning-pipeline
    
</div>

In [108]:
cross_valid = cross_validate(pipeline_multiout, 
                             X_train, y_train, 
                             cv = 5, scoring = ["accuracy"], n_jobs = -1)
print("test_accuracy:", cross_valid["test_accuracy"].mean())

test_accuracy: 0.5045935766143692


In [109]:
y_pred = cross_val_predict(pipeline_multiout, X_train, y_train, cv = 2)

In [110]:
# Посмотрим на различные метрики
print(classification_report(y_train, y_pred, zero_division = 0))

              precision    recall  f1-score   support

           0       0.78      0.28      0.41       662
           1       0.65      0.05      0.09       278
           2       0.66      0.21      0.31       473
           3       0.72      0.13      0.23       268
           4       0.00      0.00      0.00        98
           5       0.00      0.00      0.00        37
           6       0.00      0.00      0.00        18
           7       0.00      0.00      0.00        26
           8       0.00      0.00      0.00       115
           9       0.00      0.00      0.00         8
          10       0.00      0.00      0.00        84
          11       0.00      0.00      0.00        89
          12       0.61      0.17      0.27       492
          13       0.00      0.00      0.00        28
          14       0.00      0.00      0.00        57
          15       0.00      0.00      0.00        62
          16       0.00      0.00      0.00       160
          17       0.00    

<div class="alert alert-block alert-warning">

Отлично, метрики рассчитаны, но какой можно сделать вывод из полученных значений?
    
</div>

In [111]:
# Посмотрим на целевую метрику
accuracy_score(y_train, y_pred)

0.5005742176284812

<div class="alert alert-block alert-warning">

При выводе значений метрики достаточно оставить 2-3 знака после запятой. Для этого можно использовать f-строки и метод .format(). Подробнее про .format() и f-строки можно посмотреть:

https://pythonworld.ru/osnovy/formatirovanie-strok-metod-format.html https://docs-python.ru/tutorial/operatsii-tekstovymi-strokami-str-python/stroki-formatirovannye-stroki/
    
</div>

###  Тренировка окончательной модели

In [112]:
pipeline_multiout.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('vetorizer',
                                                  TfidfVectorizer(analyzer='char_wb',
                                                                  ngram_range=(1,
                                                                               3)),
                                                  'text')])),
                ('clf',
                 MultiOutputClassifier(estimator=LogisticRegression(max_iter=10000)))])

##  Предсказание и загрузка решения

In [114]:
test =  pd.read_csv(DATA_DIR / 'test.csv')

In [ ]:
pred_test = pipeline_multiout.predict(test[["text"]].astype("str"))

In [ ]:
res = pd.DataFrame(np.hstack([test["index"].values.reshape(test.shape[0], 1), pred_test]),
                  columns = ["index"]+[f"trend_id_res{i}" for i in range(50)])

In [ ]:
res.head()

In [ ]:
res.iloc[:, 1:].sum()

In [ ]:
res["trend_id_res0"].value_counts()

In [ ]:
res[["index"]+[f"trend_id_res{i}" for i in range(50)]].to_csv('/kaggle/working/submission_baseline.csv', index=False)

## Обработка текстовых признаков с помощью Sentence transformer

### Обработка тегов и текстов

В рамках данного подхода закодируем имеющиеся тэги с помощью OHE, предварительно определив все уникальные теги.
Кодировку текста в комментарии проведем с помощью семантического сравнения эмбеддингов текста с описанием каждой из 50 тем. Таким образом, мы получим коэффициент сходства темы с комментарием и отзывом.

<div class="alert alert-block alert-success">

Отличная идея 👍👍👍
</div>

In [3]:
df_trends = pd.read_csv(DATA_DIR / "trends_description.csv")
df = pd.read_csv(DATA_DIR / "train.csv")

In [4]:
sentence_model = SentenceTransformer('cointegrated/rubert-tiny2')

In [5]:
trend_embeddings = sentence_model.encode(df_trends['trend'].values, convert_to_tensor=False)
explanation_embeddings = sentence_model.encode(df_trends['explanation'].values, convert_to_tensor=False)
text_embeddings = sentence_model.encode(df['text'].values, convert_to_tensor=False)

In [156]:
text_embeddings_series = pd.Series([arr for arr in text_embeddings])

In [6]:
def clean_tags(text):
    tags = str(text).replace('{', '').replace('}', '').split(',')
    tags = [tag.strip().lower() for tag in tags]
    return tags

In [7]:
def get_ohe_tags(df, unique_tags):
    ohe_df = pd.DataFrame(index=df.index)
    
    for tag in unique_tags:
        ohe_df[tag.lower()] = df['tags'].apply(lambda x: 1 if tag in x else 0)

    new_df = pd.concat([df, ohe_df], axis=1)
    new_df.drop('tags', axis=1, inplace=True)
    return new_df

In [88]:
device = torch.device("mps")

model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

pipe_classifier = pipeline(
    "zero-shot-classification",
    model=model,
    tokenizer=tokenizer,
    framework="pt",
    device=device
)

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.huggingface.co/repos/11/df/11df887e82085c4c572ffd17b3a22b9643691659c0e120ff0d7038d632072e99/7c8e29f1115986d032e92b0fbaa0bdef1062a46f658b08705f237c05014a8541?response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1710415941&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcxMDQxNTk0MX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5odWdnaW5nZmFjZS5jby9yZXBvcy8xMS9kZi8xMWRmODg3ZTgyMDg1YzRjNTcyZmZkMTdiM2EyMmI5NjQzNjkxNjU5YzBlMTIwZmYwZDcwMzhkNjMyMDcyZTk5LzdjOGUyOWYxMTE1OTg2ZDAzMmU5MmIwZmJhYTBiZGVmMTA2MmE0NmY2NThiMDg3MDVmMjM3YzA1MDE0YTg1NDE%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=j6xRcO-6Iiin9omKe6ltRn2uUuke1yIEVQyh%7ERjVpLNBBTzehWGQairAlKoH7C6G8CcoyxbQoLhkcDDGEM4Ig9xfTtyMy%7EO6vXvnRieqZPFJIm2ISVXhH4k%7E8dQ%7EDPbfnyvy%7EVw559L0d2CTkU4GSLmpatxdx1sT%7E9ieXp8lw-q0mP4DCeoo3Gtsx%7Es2WWjGuHBbnf96-q8q%7EM70xglu%7ES5eZBAp%7EO

model.safetensors:   2%|1         | 10.5M/558M [00:00<?, ?B/s]

In [197]:
def get_encodings_and_similarity(df, text_embeddings):
    new_df = df.copy()
    text_lst = df['text'].tolist()
    labels = df_trends['trend'].tolist()

    pipe_output = pipe_classifier(
        text_lst,  # input any list of texts here
        candidate_labels=labels,
        multi_label=True,  # here you can decide if, for your task, only one hypothesis can be true, or multiple can be true
        batch_size=32  # reduce this number to 8 or 16 if you get an out-of-memory error
    )

    nli_df = pd.DataFrame(pipe_output)[['labels', 'scores']]
    
    label_scores = {}
    for index, row in nli_df.iterrows():
        labels = row['labels']
        scores = row['scores']
        
        for label, score in zip(labels, scores):
            if label in label_scores:  
                label_scores[label].append(score)
            else:
                label_scores[label] = [score]
    
    df_result = pd.DataFrame(label_scores)
    label_to_trend_id = dict(zip(df_trends['trend'], df_trends['trend_id']))

    df_result.rename(columns=label_to_trend_id, inplace=True)
    df_result_sorted = df_result.sort_index(axis=1)
    new_df[[f"nli_{i}" for i in range(50)]] = df_result_sorted
    new_df['text'] = pd.Series([arr for arr in text_embeddings])
    return new_df

In [130]:
def prepare_dataset(df):
    new_df = df.copy()
    new_df['text'].fillna('empty', inplace=True)
    new_df = new_df.drop(columns=['index', 'assessment'])    
    new_df['tags'] = new_df['tags'].apply(clean_tags)
    unique_tags=set()
    for tags in new_df['tags']:
        unique_tags.update(tags)
    if 'nan' in unique_tags: 
        unique_tags.remove('nan')
    new_df = get_ohe_tags(new_df, unique_tags)
    new_df = get_encodings_and_similarity(new_df)
    X = new_df.drop(columns=[f"trend_id_res{i}" for i in range(50)])
    y = new_df[[f"trend_id_res{i}" for i in range(50)]]
    return X, y

In [198]:
def prepare_test_dataset(df):
    new_df = df.copy()
    new_df = new_df.drop(columns=['index', 'assessment'])
    new_df['tags'] = new_df['tags'].apply(clean_tags)
    unique_tags=set()
    for tags in new_df['tags']:
        unique_tags.update(tags)
    if 'nan' in unique_tags: 
        unique_tags.remove('nan')
    new_df = get_ohe_tags(new_df, unique_tags)
    new_df = get_encodings_and_similarity(new_df, test_embeddings)
    X = new_df.copy()
    return X

<div class="alert alert-block alert-warning">

Советую при написании кода придерживаться правил оформления кода PEP8 и добавлять к функциям документацию, так называемые docstrings.

Подробнее: https://pythonworld.ru/osnovy/pep-8-rukovodstvo-po-napisaniyu-koda-na-python.html
    
</div>

In [133]:
X, y = prepare_dataset(df)

In [159]:
X['text'] = text_embeddings_series

In [161]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

In [162]:
X_train.columns

Index(['text', 'support', 'assortment', 'catalog_navigation', 'delivery',
       'promotions', 'payment', 'products_quality', 'price', 'nli_0', 'nli_1',
       'nli_2', 'nli_3', 'nli_4', 'nli_5', 'nli_6', 'nli_7', 'nli_8', 'nli_9',
       'nli_10', 'nli_11', 'nli_12', 'nli_13', 'nli_14', 'nli_15', 'nli_16',
       'nli_17', 'nli_18', 'nli_19', 'nli_20', 'nli_21', 'nli_22', 'nli_23',
       'nli_24', 'nli_25', 'nli_26', 'nli_27', 'nli_28', 'nli_29', 'nli_30',
       'nli_31', 'nli_32', 'nli_33', 'nli_34', 'nli_35', 'nli_36', 'nli_37',
       'nli_38', 'nli_39', 'nli_40', 'nli_41', 'nli_42', 'nli_43', 'nli_44',
       'nli_45', 'nli_46', 'nli_47', 'nli_48', 'nli_49'],
      dtype='object')

In [163]:
X_valid.columns

Index(['text', 'support', 'assortment', 'catalog_navigation', 'delivery',
       'promotions', 'payment', 'products_quality', 'price', 'nli_0', 'nli_1',
       'nli_2', 'nli_3', 'nli_4', 'nli_5', 'nli_6', 'nli_7', 'nli_8', 'nli_9',
       'nli_10', 'nli_11', 'nli_12', 'nli_13', 'nli_14', 'nli_15', 'nli_16',
       'nli_17', 'nli_18', 'nli_19', 'nli_20', 'nli_21', 'nli_22', 'nli_23',
       'nli_24', 'nli_25', 'nli_26', 'nli_27', 'nli_28', 'nli_29', 'nli_30',
       'nli_31', 'nli_32', 'nli_33', 'nli_34', 'nli_35', 'nli_36', 'nli_37',
       'nli_38', 'nli_39', 'nli_40', 'nli_41', 'nli_42', 'nli_43', 'nli_44',
       'nli_45', 'nli_46', 'nli_47', 'nli_48', 'nli_49'],
      dtype='object')

In [ ]:
catboost_clf = CatBoostClassifier(
    learning_rate=0.01, 
    n_estimators=5000, 
    subsample=0.075, 
    max_depth=3, 
    verbose=100,
    l2_leaf_reg = 7,
    bootstrap_type="Bernoulli",
    loss_function='MultiCrossEntropy'
)


<div class="alert alert-block alert-success">

С целью улучшения качества модели можно попробовать подобрать гиперпараметры с помощью GridSearchCV, RandomizedSearchCV и более сложный вариант использовать optuna.

Про подбор гиперпараметров дополнительно можно посмотреть: https://habr.com/ru/articles/563494/
</div>

In [190]:
catboost_clf.fit(
    X_train, 
    y_train,
    cat_features=[
        'support', 'assortment', 'catalog_navigation', 'delivery', 'promotions',
       'payment', 'products_quality', 'price'
    ],
    embedding_features=['text'],
    use_best_model=True,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=2000
)

0:	learn: 0.5646601	test: 0.5657232	best: 0.5657232 (0)	total: 81.2ms	remaining: 1m 21s
100:	learn: 0.0479082	test: 0.0518344	best: 0.0518344 (100)	total: 7.63s	remaining: 1m 7s
200:	learn: 0.0398930	test: 0.0458245	best: 0.0458245 (200)	total: 15.1s	remaining: 1m
300:	learn: 0.0358781	test: 0.0433680	best: 0.0433680 (300)	total: 22.6s	remaining: 52.4s
400:	learn: 0.0331131	test: 0.0420796	best: 0.0420796 (400)	total: 30.2s	remaining: 45.1s
500:	learn: 0.0309056	test: 0.0412783	best: 0.0412783 (500)	total: 37.7s	remaining: 37.5s
600:	learn: 0.0290569	test: 0.0408172	best: 0.0408172 (600)	total: 45.2s	remaining: 30s
700:	learn: 0.0274630	test: 0.0404454	best: 0.0404454 (700)	total: 52.8s	remaining: 22.5s
800:	learn: 0.0261418	test: 0.0402961	best: 0.0402893 (790)	total: 1m	remaining: 15s
900:	learn: 0.0249134	test: 0.0401122	best: 0.0401101 (898)	total: 1m 7s	remaining: 7.44s
999:	learn: 0.0237821	test: 0.0400256	best: 0.0400256 (999)	total: 1m 15s	remaining: 0us

bestTest = 0.040025550

In [191]:
preds = catboost_clf.predict(X_valid)

In [192]:
accuracy = accuracy_score(y_valid, preds)
print("Accuracy:", accuracy)

Accuracy: 0.5177956371986223


In [194]:
df_test = pd.read_csv(DATA_DIR / 'test.csv')

In [199]:
test_embeddings = sentence_model.encode(df_test['text'].values, convert_to_tensor=False)
test_embeddings_series = pd.Series([arr for arr in test_embeddings])

In [201]:
X_test = prepare_test_dataset(df_test)

TypeError: object of type 'float' has no len()

<div class="alert alert-block alert-danger">

Возникщую при запуске кода ошибку нужно исправить:)
    
</div>

In [ ]:
catboost_clf.fit(X, y)

In [ ]:
preds = catboost_clf.predict(X_test)

In [ ]:
res = pd.DataFrame(np.hstack([test["index"].values.reshape(df_test.shape[0], 1), preds]),
                  columns = ["index"]+[f"trend_id_res{i}" for i in range(50)])

<div class="alert alert-block alert-warning">

По итогам обучения Catboost было бы интересно рассчитать важность признаков (построить график) и проанализировать какие признаки оказались наиболее, а какие наименее значимыми.
    
</div>

In [ ]:
res[["index"]+[f"trend_id_res{i}" for i in range(50)]].to_csv(DATA_DIR /'catboost+sentence.csv', index=False)

In [ ]:
!pip install kaggle

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = ROOT_DIR / 'kaggle_account/kaggle.json'

In [ ]:
!kaggle competitions submit -c nlp-user-experience-multilabel-classification -f catboost+sentence.csv -m "Last_minute_submission"

К сожалению, до окончания дедлайна я не смог опробировать данный подход на тестовых данных.
При этом, на валидационной выборке точность использованной модели (модель градиентного бустинга от Яндекса - CatBoost) составила всего лишь 0.51, что не сильно выше baseline модели. Таким образом, данный подход можно признать неработающим.
Помимо данного подхода, в ходе работы были рассмотрены различные конфигурации нейронных сетей, но из-за спешки не были сохранены в отдельные тетрадки, а точность по ним составила еще меньше, чем базовая модель. Вместо трансофрмера был использован BERT и подход с помощью tf-idf. Которые тоже не дали значительного прироста метрики.

<div class="alert alert-block alert-success">

Отлично, итог проекта подведён)
    
По BERT рекомендую https://www.kaggle.com/code/nayansakhiya/text-classification-using-bert и https://huggingface.co/docs/transformers/model_doc/bert
    
Дополнительно можно посмотреть:
    
1. https://huggingface.co/transformers/model_doc/bert.html
2. https://colah.github.io/posts/2015-08-Understanding-LSTMs/ - Про LSTM
3. https://web.stanford.edu/~jurafsky/slp3/10.pdf - про энкодер-декодер модели, этеншены
4. https://pytorch.org/tutorials/beginner/transformer_tutorial.html - официальный гайд по трансформеру от создателей pytorch
 
</div>